In [6]:
!uv add gitsource

Resolved 171 packages in 350ms                                       
Installed 2 packages in 3ms1.3.0                                 ░░░░░░░░░░░░░░░░░░░░ [0/0] Installing wheels...                                 
 + gitsource==0.0.5
 + python-frontmatter==1.3.0


In [1]:
print("test")

test


In [2]:
## load data for hw
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)
documents = [file.parse() for file in reader.read()]

In [10]:
PREFIX="https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main"
!wget {PREFIX}/01-agentic-rag/code/rag_helper.py
!wget {PREFIX}/04-evaluation/code/evaluation_utils.py

--2026-07-12 12:02:55--  https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/01-agentic-rag/code/rag_helper.py
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.111.133, 185.199.110.133, 185.199.109.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.111.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 2134 (2.1K) [text/plain]
Saving to: ‘rag_helper.py’

rag_helper.py       100%[===================>]   2.08K  --.-KB/s    in 0s      

2026-07-12 12:02:56 (9.70 MB/s) - ‘rag_helper.py’ saved [2134/2134]

--2026-07-12 12:02:57--  https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/04-evaluation/code/evaluation_utils.py
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.111.133, 185.199.110.133, 185.199.109.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.111.133|:443... connected.
HTTP request sent, awaiting response.

In [3]:
data_gen_instructions = """
You emulate a student who is taking our LLM course.
You are given one lesson page from the course.
Formulate 5 questions this student might ask that are answered by this page.

Rules:
- The page should contain the answer to each question.
- Make the questions complete and not too short.
- Use as few words as possible from the page; don't copy its phrasing.
- The questions should resemble how people actually ask things online:
  not too formal, not too short, not too long.
- Ask about the content of the lesson, not about its formatting or filename.
""".strip()

For each page, build a JSON user prompt from its `filename` and `content`, then
call `llm_structured` with the `Questions` model. Turn each returned question
into a record labeled with the page's `filename`. The call also returns the
token usage, the same as in the lessons.



## Q1. Generating questions

Generating questions for all 72 pages costs money and takes time, so let's
start small and generate questions for just the first 3 pages:

- `01-agentic-rag/lessons/01-intro.md`
- `01-agentic-rag/lessons/02-environment.md`
- `01-agentic-rag/lessons/03-rag.md`

Each call returns the token usage, which most LLM APIs report on the response
object (e.g. `response.usage.input_tokens` / `prompt_tokens`).

What's the average number of input tokens across these 3 calls?

* 140
* 1400
* 14000
* 140000

> These numbers vary between runs, even with the same model, so pick the closest
> option. A different provider or model may land further apart, but the input
> tokens stay in the same order of magnitude - the prompt we send is the same.

ANS: 1400  

In [6]:
documents[:3] ##first 3 documents

[{'content': '# Introduction\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=rQYyFxf1FWw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn this module, we\'ll build a working Retrieval-Augmented\nGeneration (RAG) system from scratch, step by step.\n\nWe write everything in plain Python. We build a small search index by\nhand and call the LLM ourselves. I want you to see every piece first.\nThat way you know what a framework does for you before you reach for\none.\n\nPlaces where you can find me:\n\n- [My substack](https://alexeyondata.substack.com/)\n- [LinkedIn](https://www.linkedin.com/in/agrigorev/)\n- [X](https://x.com/Al_Grigor)\n\n## LLMs\n\nAn LLM (Large Language Model) is a neural network trained on massive\namounts of text. Given a prompt, it generates a continuation - a\nplausible next piece of text.\n\nThink of your phone. When you type "how are" in WhatsApp, it suggests\n"you" as the next word. "How are you" is the most common continuation.\nYour phone uses a sim

In [4]:
## generate structured response using pydantic
from pydantic import BaseModel
class Questions(BaseModel):
    questions: list[str]

In [5]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

In [9]:
import json

doc = documents[0]
user_prompt = json.dumps(doc)

In [10]:
user_prompt

'{"content": "# Introduction\\n\\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=rQYyFxf1FWw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\\n\\nIn this module, we\'ll build a working Retrieval-Augmented\\nGeneration (RAG) system from scratch, step by step.\\n\\nWe write everything in plain Python. We build a small search index by\\nhand and call the LLM ourselves. I want you to see every piece first.\\nThat way you know what a framework does for you before you reach for\\none.\\n\\nPlaces where you can find me:\\n\\n- [My substack](https://alexeyondata.substack.com/)\\n- [LinkedIn](https://www.linkedin.com/in/agrigorev/)\\n- [X](https://x.com/Al_Grigor)\\n\\n## LLMs\\n\\nAn LLM (Large Language Model) is a neural network trained on massive\\namounts of text. Given a prompt, it generates a continuation - a\\nplausible next piece of text.\\n\\nThink of your phone. When you type \\"how are\\" in WhatsApp, it suggests\\n\\"you\\" as the next word. \\"How are you\\" is the most common

In [11]:
messages = [
    {"role": "developer", "content": data_gen_instructions},
    {"role": "user", "content": user_prompt}
]

In [12]:
## use response.parse to elicit response in our pydantic structure
response = openai_client.responses.parse(
    model="gpt-5.4-mini",
    input=messages,
    text_format=Questions
)

In [13]:
response

ParsedResponse[TypeVar](id='resp_0b85169ceaab8060006a54398884e081a39ad85bddf6fbb8c8', created_at=1783904648.0, error=None, incomplete_details=None, instructions=None, metadata={}, model='gpt-5.4-mini-2026-03-17', object='response', output=[ParsedResponseOutputMessage[TypeVar](id='msg_0b85169ceaab8060006a54398926d881a39a3f7cb82c2aea19', content=[ParsedResponseOutputText[TypeVar](annotations=[], text='{"questions":["What is RAG, and how does it help when the model doesn’t know the answer itself?","Why do we treat the language model like a black box in this module instead of digging into how it works internally?","What kinds of problems with LLMs does this course say RAG can fix?","What are we building in this module to make the idea concrete?","What will the first part of the module cover, and how is the second part different?"]}', type='output_text', logprobs=[], parsed=Questions(questions=['What is RAG, and how does it help when the model doesn’t know the answer itself?', 'Why do we tr

In [14]:
result = response.output_parsed

print(result)

questions=['What is RAG, and how does it help when the model doesn’t know the answer itself?', 'Why do we treat the language model like a black box in this module instead of digging into how it works internally?', 'What kinds of problems with LLMs does this course say RAG can fix?', 'What are we building in this module to make the idea concrete?', 'What will the first part of the module cover, and how is the second part different?']


In [18]:
response.usage.input_tokens

1020

In [19]:
## do calculation for all
response_token_counts = []
for doc in documents[:3]: ##for first 3 documents 
    user_prompt = json.dumps(doc)
    messages = [
    {"role": "developer", "content": data_gen_instructions},
    {"role": "user", "content": user_prompt}
     ]
     ## use response.parse to elicit response in our pydantic structure
    response = openai_client.responses.parse(
    model="gpt-5.4-mini",
    input=messages,
    text_format=Questions)
    print(f"response input_tokens:{response.usage.input_tokens}")
    response_token_counts.append(response.usage.input_tokens)
    


response input_tokens:1020
response input_tokens:1286
response input_tokens:1753


In [22]:
import numpy as np
np.mean(response_token_counts)

np.float64(1353.0)

## Searching the chunks

We search over the same chunks as in homework 2.

Create them with `chunk_documents`:

```python
from gitsource import chunk_documents

chunks = chunk_documents(documents, size=2000, step=1000)
```

This gives 295 chunks.

Now rebuild the search from homework 2 over these chunks. Build a text index
(`Index`) and a vector index (`VectorSearch`), both keyed on `filename`. Wrap
each one in a function, `text_search` and `vector_search`, that takes a query
and the number of results to return (5 by default).

For hybrid search, reuse the `rrf` function from homework 2:

```python
def rrf(result_lists, k=60, num_results=5):
    scores = {}
    docs = {}

    for results in result_lists:
        for rank, doc in enumerate(results):
            key = (doc["filename"], doc["start"])
            scores[key] = scores.get(key, 0) + 1 / (k + rank)
            docs[key] = doc

    ranked = sorted(scores, key=scores.get, reverse=True)
    return [docs[key] for key in ranked[:num_results]]
```

Then define `hybrid_search` on top of it:

```python
def hybrid_search(query, k=60):
    text_results = text_search(query, num_results=10)
    vector_results = vector_search(query, num_results=10)
    return rrf([text_results, vector_results], k=k)
```


In [4]:
from gitsource import chunk_documents

chunks = chunk_documents(documents, size=2000, step=1000)

In [5]:
len(chunks)

295

In [6]:
def rrf(result_lists, k=60, num_results=5):
    scores = {}
    docs = {}

    for results in result_lists:
        for rank, doc in enumerate(results):
            key = (doc["filename"], doc["start"])
            scores[key] = scores.get(key, 0) + 1 / (k + rank)
            docs[key] = doc

    ranked = sorted(scores, key=scores.get, reverse=True)
    return [docs[key] for key in ranked[:num_results]]

In [7]:
def hybrid_search(query, k=60):
    text_results = text_search(query, num_results=10)
    vector_results = vector_search(query, num_results=10)
    return rrf([text_results, vector_results], k=k)

In [8]:
from minsearch import Index
from minsearch import VectorSearch

In [9]:
index = Index(text_fields=["content"],
             keyword_fields=["filename"])
index.fit(chunks)

In [10]:
def text_search(query, num_results=5):
    return index.search(query=query, num_results=num_results)

In [11]:
from embedder import Embedder
embed = Embedder()
query = "How does approximate nearest neighbor search work?"
qv = embed.encode(query)
print(qv[0])

-0.02058203437252893


In [15]:
PREFIX="https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main"
!wget {PREFIX}/cohorts/2026/04-evaluation/ground-truth.csv

--2026-07-13 18:57:38--  https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/cohorts/2026/04-evaluation/ground-truth.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.109.133, 185.199.111.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.109.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 48627 (47K) [text/plain]
Saving to: ‘ground-truth.csv’

ground-truth.csv    100%[===================>]  47.49K  --.-KB/s    in 0.02s   

2026-07-13 18:57:38 (2.86 MB/s) - ‘ground-truth.csv’ saved [48627/48627]



In [21]:
import pandas as pd
df_ground_truth = pd.read_csv("./ground-truth.csv")

In [22]:
ground_truth = df_ground_truth.to_dict(orient="records")

## Q2. First result with text search

Take the first question from the ground truth:

```python
q = ground_truth[0]["question"]
```

After running `text_search` for it, what's the `filename` of the first result?

* `01-agentic-rag/lessons/01-intro.md`
* `01-agentic-rag/lessons/03-rag.md`
* `01-agentic-rag/lessons/13-function-calling.md`
* `01-agentic-rag/lessons/10-rag-next-steps.md`


ANS:01-agentic-rag/lessons/03-rag.md

In [24]:
q = ground_truth[0]["question"]

In [25]:
text_search(q)

[{'start': 3000,
  'content': 'we drop it.\n\nBuild a prompt that includes both the question and the context:\n\n```python\nprompt = f"""\nYour task is to answer questions from the course participants\nbased on the provided context.\n\nUse the context to find relevant information and provide accurate\nanswers. If the answer is not found in the context,\nrespond with "I don\'t know."\n\nQuestion:\n{question}\n\nContext:\n{context}\n"""\n```\n\nInstead of sending the raw question to the LLM, we send this prompt:\n\n```python\nanswer = llm(prompt)\nprint(answer)\n```\n\nAfter that, the answer is correct: "Yes, you can still join. If you want to\nreceive a certificate, you need to submit your project while\nsubmissions are still open."\n\nThis is the answer we actually want to give to our students. What we\njust did is nothing but RAG.\n\n## Retrieval plus generation\n\nRAG stands for Retrieval-Augmented Generation. Generation is the LLM\nproducing text, and retrieval is search. We retriev

## Q3. First result with vector search

After running `vector_search` for the same question, what's the `filename` of
the first result?

* `01-agentic-rag/lessons/01-intro.md`
* `01-agentic-rag/lessons/03-rag.md`
* `04-evaluation/lessons/11-evaluation-intro.md`
* `04-evaluation/lessons/12-rag-answers.md`

This question was generated from `01-agentic-rag/lessons/01-intro.md`. Notice
that one method finds the right page at the top and the other doesn't. That's
exactly why we measure across the whole dataset instead of trusting one query.

ANS: '01-agentic-rag/lessons/01-intro.md'

In [26]:
len(chunks),len(documents)
chunk_fnames = []
chunk_contents = []
for chunk in chunks:
    chunk_fnames.append(chunk['filename'])
    chunk_contents.append(chunk['content'])

X = embed.encode_batch(chunk_contents)
print(X.shape)

(295, 384)


In [27]:
from minsearch import VectorSearch

In [28]:
vindex = VectorSearch(keyword_fields=['filename'])

In [29]:
vindex.fit(X,chunks) 

In [32]:
def vector_search(query, num_results=5):
    return vindex.search(embed.encode(query), num_results=num_results)

In [33]:
vector_search(q)

[{'start': 0,
  'content': '# Introduction\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=rQYyFxf1FWw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn this module, we\'ll build a working Retrieval-Augmented\nGeneration (RAG) system from scratch, step by step.\n\nWe write everything in plain Python. We build a small search index by\nhand and call the LLM ourselves. I want you to see every piece first.\nThat way you know what a framework does for you before you reach for\none.\n\nPlaces where you can find me:\n\n- [My substack](https://alexeyondata.substack.com/)\n- [LinkedIn](https://www.linkedin.com/in/agrigorev/)\n- [X](https://x.com/Al_Grigor)\n\n## LLMs\n\nAn LLM (Large Language Model) is a neural network trained on massive\namounts of text. Given a prompt, it generates a continuation - a\nplausible next piece of text.\n\nThink of your phone. When you type "how are" in WhatsApp, it suggests\n"you" as the next word. "How are you" is the most common continuation.\nYour ph


## Evaluation metrics

We evaluate search exactly as in the module, reusing the same functions from the
lecture. We change only the label. Our ground truth uses `filename`, so a result
counts as a hit when a returned chunk's `filename` matches the question's
`filename`, not a document `id`.

As a reminder, these functions do the following:

- `compute_relevance` runs search for a question and returns a list of 0s and 1s
- `hit_rate` is the fraction of questions where the correct page appears in the results
- `mrr` (Mean Reciprocal Rank) also rewards finding the page near the top
- `evaluate` runs a search function over the whole ground truth and returns both metrics

## Q4. Evaluating text search

Evaluate `text_search` on the ground truth data.

What's the Hit Rate?

* 0.55
* 0.66
* 0.76
* 0.88

ANS: 0.76

In [39]:
def compute_relevance_text(q):
    doc_id = q["filename"]
    results = text_search(query=q["question"])

    relevance = []
    for d in results:
        relevance.append(int(d["filename"] == doc_id))

    return relevance

In [46]:
## make it generic by passsing "function_name" as param -> kw search/vec search/hybrid search
def compute_relevance(q, search_function):
    doc_id = q["filename"]
    results = search_function(query=q["question"])

    relevance = []
    for d in results:
        relevance.append(int(d["filename"] == doc_id))

    return relevance

In [47]:
## do for full list of GT
def compute_relevance_total(ground_truth, search_function):
    relevance_total = []

    for q in tqdm(ground_truth):
        relevance = compute_relevance(q, search_function)
        relevance_total.append(relevance)

    return relevance_total

In [48]:
def hit_rate(relevance):
    cnt = 0

    for line in relevance:
        if 1 in line:
            cnt = cnt + 1

    return cnt / len(relevance)

In [49]:
def mrr(relevance):
    total_score = 0.0

    for line in relevance:
        for rank in range(len(line)):
            if line[rank] == 1:
                total_score = total_score + 1 / (rank + 1)
                break

    return total_score / len(relevance)

In [50]:
def evaluate(ground_truth, search_function):
    relevance_total = compute_relevance_total(ground_truth, search_function)

    return {
        "hit_rate": hit_rate(relevance_total),
        "mrr": mrr(relevance_total),
    }

In [51]:
from tqdm.auto import tqdm
evaluate(
    ground_truth,
    text_search
)

  0%|          | 0/360 [00:00<?, ?it/s]

{'hit_rate': 0.7583333333333333, 'mrr': 0.5942592592592594}


## Q5. Evaluating vector search

Now evaluate `vector_search` - the part we left for the homework, since the
module only evaluated keyword search.

What's the MRR?

* 0.35
* 0.45
* 0.55
* 0.65

ANS: 0.55

In [52]:
from tqdm.auto import tqdm
evaluate(
    ground_truth,
    vector_search
)

  0%|          | 0/360 [00:00<?, ?it/s]

{'hit_rate': 0.725, 'mrr': 0.5486111111111112}

## Q6. Tuning hybrid search

The `k` constant in RRF controls how much the top ranks matter. A smaller `k`
sharpens the gap between positions, so being at the top of a list counts for
more. The RRF paper uses 60 as a default, but the best value depends on the data
- so let's measure it.

Evaluate `hybrid_search` over the full ground truth dataset for `k` values 1,
50, 100, and 200. Compare the MRR values for these runs.

Which `k` gives the best MRR?

* 1
* 50
* 100
* 200

> Several values of `k` may give the same MRR. If there's a tie, pick the
> smallest `k`.

## Using this framework

You now have an `evaluate` function that takes any search function and returns
Hit Rate and MRR.

Use it to measure any change you make to search:

- tune the field boosts in keyword search
- try a different embedding model for vector search
- change `k` in the RRF formula for hybrid search
- change the number of results you return

Change a setting, re-run `evaluate`, and see whether the metric moves. The
ground truth stays fixed, so the comparison is fair. That's how you replace
guessing with measuring.

ANS: 1

In [ ]:
k_range = [1,50,100,200]

In [55]:
from tqdm.auto import tqdm

k_range = [1, 50, 100, 200]

results = {}

for k in tqdm(k_range):
    score = evaluate(
        ground_truth,
        lambda query, k=k: hybrid_search(query, k=k)
    )
    results[k] = score

print(results)

  0%|          | 0/4 [00:00<?, ?it/s]

  0%|          | 0/360 [00:00<?, ?it/s]

  0%|          | 0/360 [00:00<?, ?it/s]

  0%|          | 0/360 [00:00<?, ?it/s]

  0%|          | 0/360 [00:00<?, ?it/s]

{1: {'hit_rate': 0.8388888888888889, 'mrr': 0.6481944444444449}, 50: {'hit_rate': 0.8361111111111111, 'mrr': 0.637916666666667}, 100: {'hit_rate': 0.8361111111111111, 'mrr': 0.637916666666667}, 200: {'hit_rate': 0.8361111111111111, 'mrr': 0.637916666666667}}


In [56]:

sorted_results = dict(
    sorted(results.items(), key=lambda x: x[1]['mrr'], reverse=True)
)

print(sorted_results)

{1: {'hit_rate': 0.8388888888888889, 'mrr': 0.6481944444444449}, 50: {'hit_rate': 0.8361111111111111, 'mrr': 0.637916666666667}, 100: {'hit_rate': 0.8361111111111111, 'mrr': 0.637916666666667}, 200: {'hit_rate': 0.8361111111111111, 'mrr': 0.637916666666667}}
